# Churros - RTL del semaforo (explicacion interactiva)

Cantidad de la **Capsula 2** del flujo. Aqui no solo corres el codigo: lo
**entiendes bloque por bloque**, lo ves **animado** y **juegas** con el.

El RTL describe el *que* (el comportamiento), todavia no el silicio.

> Corre las celdas en orden (Shift+Enter).

## 1 - Preparar el entorno (instalar + clonar el repo)

In [ ]:
import os
!apt-get -qq update && apt-get -qq install -y iverilog > /dev/null
REPO_URL = 'https://github.com/Misterio12947/galletas-y-postres-chips.git'
REPO = 'galletas-y-postres-chips'
if not os.path.isdir(REPO):
    !git clone --depth 1 $REPO_URL
else:
    !cd $REPO && git pull --ff-only
RTL = f'{REPO}/common/rtl/semaphore/rtl/semaphore.v'
assert os.path.exists(RTL), 'No encuentro el RTL'
print('Listo. RTL en:', RTL)

## 2 - El semaforo es una **maquina de estados**

Un semaforo solo puede estar en uno de tres estados, y SIEMPRE avanza en el
mismo orden, en anillo:

**VERDE -> AMARILLO -> ROJO -> VERDE -> ...**

Cada estado dura cierto numero de *ticks* de reloj. Eso es exactamente lo que
describe el RTL. Veamoslo moverse (dale al boton de play):

In [ ]:
import numpy as np, matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Circle
from IPython.display import HTML

# Modelo en Python del semaforo (SOLO para esta animacion visual)
T = {'VERDE': 5, 'AMARILLO': 2, 'ROJO': 5}
orden = ['VERDE', 'AMARILLO', 'ROJO']
seq = []
for _ in range(2):                 # dos vueltas completas
    for st in orden:
        seq += [st] * T[st]

col = {'VERDE': '#2ecc71', 'AMARILLO': '#f1c40f', 'ROJO': '#e74c3c'}
off = '#3a3a3a'

fig, (axL, axR) = plt.subplots(1, 2, figsize=(9, 4))

# Izquierda: el semaforo fisico (3 focos)
axL.set_xlim(0, 2); axL.set_ylim(0, 4); axL.axis('off')
axL.add_patch(plt.Rectangle((0.5, 0.2), 1, 3.6, fc='#222', ec='k'))
focos = {'ROJO': Circle((1, 3.1), 0.35, fc=off),
         'AMARILLO': Circle((1, 2.0), 0.35, fc=off),
         'VERDE': Circle((1, 0.9), 0.35, fc=off)}
for c in focos.values(): axL.add_patch(c)

# Derecha: el diagrama de estados (3 nodos en anillo)
axR.set_xlim(-1.6, 1.6); axR.set_ylim(-1.6, 1.6); axR.axis('off')
pos = {'VERDE': (0, 1), 'AMARILLO': (1, -0.7), 'ROJO': (-1, -0.7)}
nodos = {}
for st, (x, y) in pos.items():
    nodos[st] = Circle((x, y), 0.38, fc=off, ec='w')
    axR.add_patch(nodos[st])
    axR.text(x, y, st[0], ha='center', va='center', color='w', weight='bold')
for a, b in [('VERDE','AMARILLO'), ('AMARILLO','ROJO'), ('ROJO','VERDE')]:
    (x1,y1),(x2,y2) = pos[a], pos[b]
    axR.annotate('', xy=(x2,y2), xytext=(x1,y1),
                 arrowprops=dict(arrowstyle='->', color='gray', shrinkA=20, shrinkB=20))

titulo = fig.suptitle('')
def draw(i):
    st = seq[i]
    for n, c in focos.items(): c.set_fc(col[n] if n == st else off)
    for n, d in nodos.items(): d.set_fc(col[n] if n == st else off)
    titulo.set_text(f'tick {i}  -  estado: {st}')
    return []

anim = FuncAnimation(fig, draw, frames=len(seq), interval=350)
plt.close(fig)
HTML(anim.to_jshtml())

## 3 - El codigo real, de principio a fin

Este es el archivo que simulamos (y que corre igual en Icarus, VCS y Quartus).
Lo leemos directo del repo:

In [ ]:
print(open(RTL).read())

## 4 - Bloque por bloque

### Bloque A - Codificacion de estados
Le damos un numero a cada estado. Dos bits alcanzan para tres estados.

```verilog
localparam [1:0] S_GREEN  = 2'd0,
                 S_YELLOW = 2'd1,
                 S_RED    = 2'd2;
```

### Bloque B - Logica de siguiente estado (el anillo)
Puramente combinacional: dado el estado actual, cual sigue. Es el anillo
verde -> amarillo -> rojo -> verde que viste en la animacion.

```verilog
always @(*) begin
    case (state)
        S_GREEN:  next_state = S_YELLOW;
        S_YELLOW: next_state = S_RED;
        S_RED:    next_state = S_GREEN;
        default:  next_state = S_GREEN;
    endcase
end
```

### Bloque C - Registro de estado + contador
Lo secuencial (con el reloj). El contador mide cuantos ticks llevamos; cuando
llega al limite del estado, **avanza** al siguiente y reinicia la cuenta. El
`rst_n` lo regresa todo a VERDE.

```verilog
always @(posedge clk or negedge rst_n) begin
    if (!rst_n) begin
        state <= S_GREEN; cnt <= 0;
    end else if (cnt == limit - 1) begin
        state <= next_state; cnt <= 0;   // se acabo el tiempo: avanza
    end else begin
        cnt <= cnt + 1;                  // sigue contando
    end
end
```

### Bloque D - Salidas (Moore)
Las luces dependen SOLO del estado (eso es una maquina de Moore). Por diseno,
exactamente una luz encendida a la vez -> justo lo que el testbench verifica.

```verilog
always @(*) begin
    red = 0; yellow = 0; green = 0;
    case (state)
        S_GREEN:  green  = 1;
        S_YELLOW: yellow = 1;
        S_RED:    red    = 1;
    endcase
end
```

## 5 - Experimenta (con el RTL de verdad)

Mueve los sliders: cambian las duraciones, se **re-simula el RTL real** con
Icarus y se dibuja la onda. Observa como la propiedad *una sola luz a la vez*
se mantiene siempre.

In [ ]:
import subprocess

def _parse_vcd(path, names):
    sym2name, changes, t = {}, {n: [] for n in names}, 0
    for ln in open(path):
        ln = ln.strip()
        if ln.startswith('$var'):
            p = ln.split(); sym2name[p[3]] = p[4]
        elif ln.startswith('#'):
            t = int(ln[1:])
        elif ln and ln[0] in '01xz' and len(ln) >= 2:
            nm = sym2name.get(ln[1:])
            if nm in names: changes[nm].append((t, 1 if ln[0]=='1' else 0))
    return changes

def run_sim(verde=3, amarillo=2, rojo=3, total=500):
    tb = f'''`timescale 1ns/1ps
module tb;
  reg clk=0, rst_n=0; wire red, yellow, green;
  semaphore #(.T_GREEN({verde}), .T_YELLOW({amarillo}), .T_RED({rojo}), .CW(8)) dut(
    .clk(clk), .rst_n(rst_n), .red(red), .yellow(yellow), .green(green));
  always #5 clk = ~clk;
  integer errors = 0;
  always @(posedge clk) if (rst_n) if ((red+yellow+green)!==1) errors=errors+1;
  initial begin
    $dumpfile("d.vcd"); $dumpvars(0, tb);
    #12 rst_n=1; #{total};
    if (errors==0) $display(">>> PASS: 1 luz a la vez"); else $display(">>> FAIL %0d", errors);
    $finish;
  end
endmodule
'''
    open('tb_gen.v','w').write(tb)
    r = subprocess.run(['bash','-lc', f'iverilog -g2012 -o s {RTL} tb_gen.v && vvp s'],
                       capture_output=True, text=True)
    print((r.stdout + r.stderr).strip())
    ch = _parse_vcd('d.vcd', ['green','yellow','red'])
    tmax = max((tv[-1][0] for tv in ch.values() if tv), default=1) + 10
    fig, axes = plt.subplots(3, 1, figsize=(10, 2.4), sharex=True)
    for ax, nm, c in zip(axes, ['green','yellow','red'], ['tab:green','gold','tab:red']):
        xs, ys = [0], [0]
        for tt, vv in ch[nm]: xs.append(tt); ys.append(vv)
        xs.append(tmax); ys.append(ys[-1])
        ax.step(xs, ys, where='post', color=c, lw=2)
        ax.set_ylim(-0.2,1.2); ax.set_yticks([0,1])
        ax.set_ylabel(nm, rotation=0, ha='right', va='center')
    axes[-1].set_xlabel('tiempo (ns)'); plt.tight_layout(); plt.show()

from ipywidgets import interact, IntSlider
interact(run_sim, verde=IntSlider(3,1,10), amarillo=IntSlider(2,1,10),
         rojo=IntSlider(3,1,10), total=IntSlider(500,200,1000,100));

## 6 - Para pensar

- Si quitas el `rst_n`, con que estado arranca el semaforo? (pista: mira el bloque C)
- Por que NUNCA se encienden dos luces juntas? (pista: bloque D, maquina de Moore)
- Que cambiaria si fuera una maquina de **Mealy** en vez de Moore?

Lo siguiente en el flujo: **verificacion funcional** - comprobar con un
testbench que esto hace lo correcto, para todos los casos que se nos ocurran.